# Day 1 - Module 2: Prompt Engineering for Agents
## Hexaware Technologies - Practical Agentic AI Training

This notebook covers:
1. **Zero-Shot vs Few-Shot Prompting** on Hexaware incident telemetry.
2. **Chain-of-Thought (CoT)** diagnostic reasoning.
3. **ReAct (Reason + Act)** pattern for agentic decision-making.
4. **Function Injection**: Structuring strict JSON tool schemas.


In [1]:
import os
import json
import re
from dotenv import load_dotenv
import ollama

load_dotenv()
MODEL_NAME = os.getenv("OLLAMA_MODEL", "llama3.2:1b")
SAMPLE_LOG = "2026-08-24 14:02:11 [ERR_DB_POOL_EXHAUSTED] Hexaware Cloud DB Node 4: active_tx=0, idle_conn=250, max_limit=200. Latency spike: 4200ms."

print(f"Using Model: [{MODEL_NAME}]")
print(f"Sample Telemetry Log: {SAMPLE_LOG}")


Using Model: [llama3.2:1b]
Sample Telemetry Log: 2026-08-24 14:02:11 [ERR_DB_POOL_EXHAUSTED] Hexaware Cloud DB Node 4: active_tx=0, idle_conn=250, max_limit=200. Latency spike: 4200ms.


### Step 1: Zero-Shot Prompting


In [2]:
zero_shot_messages = [
    {"role": "system", "content": "You are an IT support assistant."},
    {"role": "user", "content": f"Analyze this incident log and identify what is wrong:\n{SAMPLE_LOG}"}
]
res_zero = ollama.chat(model=MODEL_NAME, messages=zero_shot_messages, options={"temperature": 0.2})
print(res_zero["message"]["content"].strip())


Let's analyze the incident log.

The error message indicates that the Hexaware Cloud DB Node 4 has experienced a pool exhaustion issue, which is a critical problem in a distributed database system. Here's a breakdown of the log entries:

1. `2026-08-24 14:02:11 [ERR_DB_POOL_EXHAUSTED]`: This is the error message itself, indicating that the pool has exhausted its available connections.
2. `Hexaware Cloud DB Node 4`: This is the specific node that is experiencing the issue.
3. `active_tx=0, idle_conn=250, max_limit=200`: These are the current values of the node's active transactions, idle connections, and maximum available connections, respectively.
4. `Latency spike: 4200ms`: This is the duration of the latency spike, which is a significant increase in response time.

The issue is that the node has reached its maximum available connections (200), and the active transactions have decreased to 0, indicating that the node is not being used. This can lead to a range of problems, including:


### Step 2: Few-Shot Prompting (Persona + Strict Output Schema)


In [4]:
few_shot_messages = [
    {"role": "system", "content": """You are an elite Hexaware Site Reliability Engineer (SRE).
Classify the error code, determine severity (Sev 1 / Sev 2 / Sev 3), root cause, and immediate remediation action.
Format strictly as:
SEVERITY: <Sev 1/2/3>
ROOT CAUSE: <one sentence>
ACTION: <remediation command>"""},
    {"role": "user", "content": "2026-08-20 09:12:00 [ERR_AUTH_EXPIRED] Token expired for user hex_admin_01."},
    {"role": "assistant", "content": "SEVERITY: Sev 3\nROOT CAUSE: Session authentication token exceeded TTL limit.\nACTION: Refresh OAuth token cache and request user re-authentication."},
    {"role": "user", "content": f"Analyze this incident log:\n{SAMPLE_LOG}"}
]
res_few = ollama.chat(model=MODEL_NAME, messages=few_shot_messages, options={"temperature": 0.1})
print(res_few["message"]["content"].strip())


SEVERITY: Sev 2
ROOT CAUSE: High latency spike in database pool, indicating a potential bottleneck or resource contention issue.
ACTION: Increase the number of available database nodes or optimize database connection pooling configuration to reduce latency.


### Step 3: Chain-of-Thought (CoT) Prompting


In [ ]:
cot_messages = [
    {"role": "system", "content": "You are an expert Hexaware Cloud Architect. Think step-by-step before arriving at the conclusion."},
    {"role": "user", "content": f"""Analyze this incident log: {SAMPLE_LOG}
Please structure your thinking as:
Step 1 - Deconstruct the metrics (active transactions vs idle connections vs threshold).
Step 2 - Identify the bottleneck or failure mode.
Step 3 - Propose the exact architectural solution."""}
]
res_cot = ollama.chat(model=MODEL_NAME, messages=cot_messages, options={"temperature": 0.2})
print(res_cot["message"]["content"].strip())


Analyzing the incident log, I'll break down the steps to understand the issue and propose a solution.

**Step 1: Deconstruct the metrics**

*   `active_tx=0`: This suggests that there are no active transactions currently running on the database node.
*   `idle_conn=250`: This indicates that the node has 250 idle connections, which is a relatively high number. This could be a sign of a resource-intensive process or a high volume of connections.
*   `max_limit=200`: This sets a maximum limit for the number of connections that can be idle at the same time. If the number of idle connections exceeds this limit, it could lead to a "pool exhausted" error.

**Step 2: Identify the bottleneck or failure mode**

*   The "pool exhausted" error typically occurs when the number of active transactions exceeds the maximum limit set by the database node. In this case, the active transactions are at 0, which means there are no transactions running.
*   The high number of idle connections (250) suggests 

: 

### Step 4: ReAct (Reason + Act) & JSON Tool Injection


In [6]:
system_prompt = """You are HexaAgent, an autonomous remediation agent at Hexaware Technologies.
You have access to the following tools:
- restart_connection_pool_manager(node_id: str): Restarts the idle pool manager on the specified node.
- escalate_to_oncall(severity: str, reason: str): Escalates incident to human on-call commander.
- query_service_health(service_name: str): Checks real-time health metrics of a cloud service.

When given an incident, follow the ReAct framework:
Thought: Explain your reasoning step.
Action: Output a single valid JSON object containing "tool" and "parameters".

Example Action:
```json
{
  "tool": "restart_connection_pool_manager",
  "parameters": {"node_id": "node_4"}
}
```
Do not include commentary outside the Thought and Action JSON block."""

user_query = f"High alert received: {SAMPLE_LOG}. Take autonomous action according to SOP."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_query}
]

response = ollama.chat(model=MODEL_NAME, messages=messages, options={"temperature": 0.1})
raw_text = response["message"]["content"]
print("Agent Raw Output:\n" + raw_text)

# Extract & Validate JSON Schema
json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw_text, re.DOTALL)
json_str = json_match.group(1) if json_match else raw_text
try:
    payload = json.loads(json_str)
    print("\n[SUCCESS] Valid JSON Action Schema:")
    print(json.dumps(payload, indent=2))
except Exception as e:
    print(f"JSON validation error: {e}")


Agent Raw Output:
```
{
  "tool": "restart_connection_pool_manager",
  "parameters": {"node_id": "node_4"}
}
```

[SUCCESS] Valid JSON Action Schema:
{
  "tool": "restart_connection_pool_manager",
  "parameters": {
    "node_id": "node_4"
  }
}
